# [Super AI Engineer Season 6] Hackathon Week 6
## 5 Domains Hackathon: Heart Disease Prediction

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Heart Disease Prediction
- Notebook: No-restart tabular ensemble pipeline
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Configuration  
3. Data Loading & Initial Inspection  
4. Target Normalization & Feature Engineering  
5. Train/Validation Split  
6. Preprocessing & Model Candidates  
7. Model Training & Threshold Tuning  
8. Refit & Test Prediction  
9. Prediction & Submission Generation

# 1. Setup & Imports
### 1.1 Import stable no-restart dependencies

This version avoids forced package installation and uses libraries that are normally available in Kaggle or Colab first.

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Configuration
### 2.1 Set paths and experiment controls

Keep dataset paths, target settings, validation size, random seed, and output paths together so the notebook is easy to adjust.

In [ ]:
TARGET_COL = "History of HeartDisease or Attack"
POSITIVE_LABEL = "Yes"
NEGATIVE_LABEL = "No"
COMPETITION_NAME = "super-ai-engineer-ss-6-individual-heart-disease-prediction"
LEGACY_COMPETITION_NAME = "super-ai-engineer-ss-6-heart-disease-prediction"

VALID_FRAC = float(os.environ.get("HEART_VALID_FRAC", "0.18"))
MAX_TRAIN_ROWS = int(os.environ.get("HEART_MAX_TRAIN_ROWS", "0"))  # 0 = use all rows
USE_OPTIONAL_MODELS = os.environ.get("HEART_USE_OPTIONAL_MODELS", "1") == "1"

DATA_DIR_CANDIDATES = []
if os.environ.get("HEART_DATA_DIR"):
    DATA_DIR_CANDIDATES.append(Path(os.environ["HEART_DATA_DIR"]))
DATA_DIR_CANDIDATES.extend([
    Path(f"/kaggle/input/competitions/{COMPETITION_NAME}"),
    Path(f"/kaggle/input/{COMPETITION_NAME}"),
    Path(f"/kaggle/input/competitions/{LEGACY_COMPETITION_NAME}"),
    Path(f"/kaggle/input/{LEGACY_COMPETITION_NAME}"),
    Path.cwd() / COMPETITION_NAME,
    Path.cwd() / LEGACY_COMPETITION_NAME,
])

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SUBMISSION_PATH = WORK_DIR / "heart_disease_lv2_submission.csv"
VALIDATION_REPORT_PATH = WORK_DIR / "heart_disease_lv2_validation_report.csv"


def resolve_data_dir(candidates):
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists() and (path / "sample_submission.csv").exists():
            return path
    print("Checked dataset candidates:")
    for path in candidates:
        print(" -", path)
    raise FileNotFoundError("Dataset not found. Attach the Kaggle dataset or set HEART_DATA_DIR.")

DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)
print("VALID_FRAC:", VALID_FRAC)
print("USE_OPTIONAL_MODELS:", USE_OPTIONAL_MODELS)

# 3. Data Loading & Initial Inspection
### 3.1 Load train, test, and sample submission

Read the input tables and inspect the schema, missing values, and target representation before preprocessing.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_sub.shape)
print("Columns:", train_df.columns.tolist())
display(train_df.head())
display(sample_sub.head())

if TARGET_COL not in train_df.columns:
    raise KeyError(f"Target column not found: {TARGET_COL}")

# 4. Target Normalization & Feature Engineering
### 4.1 Normalize labels and add robust tabular features

Convert the target into one consistent representation and add simple risk-related features while keeping the original columns available to the model pipeline.

In [ ]:
YES_NO_MAP = {
    "yes": POSITIVE_LABEL, "y": POSITIVE_LABEL, "true": POSITIVE_LABEL, "1": POSITIVE_LABEL, "1.0": POSITIVE_LABEL,
    "no": NEGATIVE_LABEL, "n": NEGATIVE_LABEL, "false": NEGATIVE_LABEL, "0": NEGATIVE_LABEL, "0.0": NEGATIVE_LABEL,
}
YES_NO_NUMERIC_MAP = {
    "yes": 1, "y": 1, "true": 1, "1": 1, "1.0": 1,
    "no": 0, "n": 0, "false": 0, "0": 0, "0.0": 0,
}

BINARY_COLUMNS = [
    "High Blood Pressure",
    "Told High Cholesterol",
    "Cholesterol Checked",
    "Smoked 100+ Cigarettes",
    "Diagnosed Stroke",
    "Diagnosed Diabetes",
    "Leisure Physical Activity",
    "Heavy Alcohol Consumption",
    "Health Care Coverage",
    "Doctor Visit Cost Barrier",
    "Difficulty Walking",
]

FRUIT_VEG_CANDIDATES = [
    "Vegetable or Fruit Intake (1+ per Day)",
    "Fruit or Vegetable Intake (1+ per Day)",
    "Fruit or Vegetable Intake (1+ per Day)'",
]

GENERAL_HEALTH_MAP = {
    "excellent": 5,
    "very good": 4,
    "verygood": 4,
    "good": 3,
    "fair": 2,
    "poor": 1,
}


def normalize_target_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return POSITIVE_LABEL if float(value) >= 0.5 else NEGATIVE_LABEL
    key = str(value).strip().lower()
    return YES_NO_MAP.get(key, str(value).strip())


def yes_no_to_numeric(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    key = series.astype(str).str.strip().str.lower()
    return key.map(YES_NO_NUMERIC_MAP)


def first_existing_column(df: pd.DataFrame, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in BINARY_COLUMNS:
        if col in df.columns:
            df[f"{col}__bin"] = yes_no_to_numeric(df[col])

    fruit_veg_col = first_existing_column(df, FRUIT_VEG_CANDIDATES)
    if fruit_veg_col is not None:
        df["FruitVeg_Intake__bin"] = yes_no_to_numeric(df[fruit_veg_col])

    if "Body Mass Index" in df.columns:
        bmi = pd.to_numeric(df["Body Mass Index"], errors="coerce")
        df["BMI_missing"] = bmi.isna().astype(int)
        df["BMI_squared"] = bmi ** 2
        df["BMI_log1p"] = np.log1p(bmi.clip(lower=0))
        df["BMI Category"] = pd.cut(
            bmi,
            bins=[-np.inf, 18.5, 25, 30, np.inf],
            labels=["Underweight", "Normal", "Overweight", "Obese"],
        ).astype("object")

    if "Age" in df.columns:
        age = pd.to_numeric(df["Age"], errors="coerce")
        df["Age_num"] = age
        df["Age_squared"] = age ** 2

    if "Education Level" in df.columns:
        df["Education_num"] = pd.to_numeric(df["Education Level"], errors="coerce")
    if "Income Level" in df.columns:
        df["Income_num"] = pd.to_numeric(df["Income Level"], errors="coerce")

    if "General Health" in df.columns:
        if pd.api.types.is_numeric_dtype(df["General Health"]):
            df["General Health_num"] = pd.to_numeric(df["General Health"], errors="coerce")
        else:
            df["General Health_num"] = df["General Health"].astype(str).str.strip().str.lower().map(GENERAL_HEALTH_MAP)

    if "Body Mass Index" in df.columns and "Age_num" in df.columns:
        df["BMI_x_Age"] = pd.to_numeric(df["Body Mass Index"], errors="coerce") * df["Age_num"]

    risk_cols = [
        "High Blood Pressure__bin",
        "Told High Cholesterol__bin",
        "Smoked 100+ Cigarettes__bin",
        "Diagnosed Stroke__bin",
        "Diagnosed Diabetes__bin",
        "Difficulty Walking__bin",
        "Heavy Alcohol Consumption__bin",
    ]
    existing_risk_cols = [col for col in risk_cols if col in df.columns]
    if existing_risk_cols:
        df["Risk_Factor_Count"] = df[existing_risk_cols].sum(axis=1, min_count=1)

    metabolic_cols = [col for col in ["High Blood Pressure__bin", "Told High Cholesterol__bin", "Diagnosed Diabetes__bin"] if col in df.columns]
    if metabolic_cols:
        df["Metabolic_Risk_Count"] = df[metabolic_cols].sum(axis=1, min_count=1)

    if "Body Mass Index" in df.columns and "Leisure Physical Activity__bin" in df.columns:
        bmi = pd.to_numeric(df["Body Mass Index"], errors="coerce")
        df["Obesity_No_Activity_Risk"] = ((bmi >= 30) & (df["Leisure Physical Activity__bin"] == 0)).astype(int)

    return df

train_df = train_df.copy()
train_df[TARGET_COL] = train_df[TARGET_COL].map(normalize_target_value)
train_df = train_df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

train_features = create_features(train_df)
test_features = create_features(test_df)

if MAX_TRAIN_ROWS > 0 and len(train_features) > MAX_TRAIN_ROWS:
    train_features = train_features.sample(MAX_TRAIN_ROWS, random_state=SEED).reset_index(drop=True)

print("Train after cleaning:", train_features.shape)
print("Test after features:", test_features.shape)
display(train_features[TARGET_COL].value_counts())
display(train_features.head(3))

# 5. Train/Validation Split
### 5.1 Use a stratified validation split

Keep the class ratio similar between train and validation, then use validation only for model selection and threshold tuning.

In [ ]:
y = train_features[TARGET_COL].astype(str)
X = train_features.drop(columns=[TARGET_COL])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=VALID_FRAC,
    random_state=SEED,
    stratify=y,
)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape)
print("Train target distribution:")
display(y_train.value_counts(normalize=True).to_frame("ratio"))
print("Valid target distribution:")
display(y_valid.value_counts(normalize=True).to_frame("ratio"))

# 6. Preprocessing & Model Candidates
### 6.1 Build preprocessing and candidate models

Handle numeric and categorical features consistently, then prepare several model candidates that can run without extra installation.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X_df: pd.DataFrame):
    numeric_cols = X_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [col for col in X_df.columns if col not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ], remainder="drop")
    return preprocessor, numeric_cols, categorical_cols

preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train)
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Categorical sample:", categorical_cols[:10])

class_weight = {NEGATIVE_LABEL: 1.0, POSITIVE_LABEL: 4.0}
models = {}

try:
    # Keep HistGradient without class_weight for cross-version stability.
    # Recall is controlled later by validation threshold tuning.
    hgb_params = dict(
        max_iter=450,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        early_stopping=True,
        random_state=SEED,
    )
    models["hist_gbdt"] = HistGradientBoostingClassifier(**hgb_params)
except Exception as exc:
    print("Skip HistGradientBoosting:", repr(exc))

models["extra_trees"] = ExtraTreesClassifier(
    n_estimators=450,
    min_samples_leaf=12,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)

models["logistic_balanced"] = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=1500, C=0.7, class_weight=class_weight, n_jobs=-1, random_state=SEED)),
])

if USE_OPTIONAL_MODELS:
    try:
        from lightgbm import LGBMClassifier
        models["lightgbm"] = LGBMClassifier(
            n_estimators=900,
            learning_rate=0.025,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary",
            class_weight=class_weight,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1,
        )
        print("Optional model enabled: lightgbm")
    except Exception as exc:
        print("LightGBM unavailable:", repr(exc))

    try:
        from xgboost import XGBClassifier
        pos_count = max((y_train == POSITIVE_LABEL).sum(), 1)
        neg_count = max((y_train == NEGATIVE_LABEL).sum(), 1)
        models["xgboost"] = XGBClassifier(
            n_estimators=650,
            learning_rate=0.035,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=float(neg_count / pos_count),
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
        )
        print("Optional model enabled: xgboost")
    except Exception as exc:
        print("XGBoost unavailable:", repr(exc))

print("Models:", list(models.keys()))

# 7. Model Training & Threshold Tuning
### 7.1 Train candidates and tune the decision threshold

Evaluate each model on validation data, tune the threshold for the target metric, and blend the strongest probability outputs.

In [ ]:
def positive_probability(estimator, X_df):
    proba = estimator.predict_proba(X_df)
    classes = list(estimator.classes_) if hasattr(estimator, "classes_") else list(estimator.named_steps["model"].classes_)
    if POSITIVE_LABEL in classes:
        idx = classes.index(POSITIVE_LABEL)
    elif 1 in classes:
        idx = classes.index(1)
    elif 1.0 in classes:
        idx = classes.index(1.0)
    else:
        idx = 1 if proba.shape[1] > 1 else 0
    return proba[:, idx]


def f2_score_from_scores(y_true, scores, threshold):
    pred = np.where(scores >= threshold, POSITIVE_LABEL, NEGATIVE_LABEL)
    return float(fbeta_score(y_true, pred, beta=2.0, pos_label=POSITIVE_LABEL))


def tune_threshold(y_true, scores):
    thresholds = np.linspace(0.02, 0.98, 193)
    values = [f2_score_from_scores(y_true, scores, t) for t in thresholds]
    best_idx = int(np.argmax(values))
    return float(thresholds[best_idx]), float(values[best_idx])

trained_models = {}
valid_scores_by_model = {}
report_rows = []

for name, model in models.items():
    print("\nTraining", name)
    estimator = Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])
    try:
        estimator.fit(X_train, y_train)
        scores = positive_probability(estimator, X_valid)
        threshold, f2_value = tune_threshold(y_valid, scores)
    except Exception as exc:
        print("Skip failed model", name, "->", repr(exc))
        continue
    trained_models[name] = estimator
    valid_scores_by_model[name] = scores
    report_rows.append({"model": name, "best_threshold": threshold, "valid_f2": f2_value})
    print(f"{name}: F2={f2_value:.5f}, threshold={threshold:.3f}")

if not trained_models:
    raise RuntimeError("No model was trained.")

valid_score_matrix = np.vstack([valid_scores_by_model[name] for name in trained_models])
blend_scores = valid_score_matrix.mean(axis=0)
blend_threshold, blend_f2 = tune_threshold(y_valid, blend_scores)
report_rows.append({"model": "blend_mean", "best_threshold": blend_threshold, "valid_f2": blend_f2})

report_df = pd.DataFrame(report_rows).sort_values("valid_f2", ascending=False).reset_index(drop=True)
report_df.to_csv(VALIDATION_REPORT_PATH, index=False)
display(report_df)

best_model_name = report_df.iloc[0]["model"]
print("Best validation choice:", best_model_name)
print("Blend threshold:", blend_threshold, "Blend F2:", blend_f2)

blend_pred = np.where(blend_scores >= blend_threshold, POSITIVE_LABEL, NEGATIVE_LABEL)
print("Blend classification report:")
print(classification_report(y_valid, blend_pred, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_valid, blend_pred), index=[NEGATIVE_LABEL, POSITIVE_LABEL], columns=[NEGATIVE_LABEL, POSITIVE_LABEL]))

# 8. Refit & Test Prediction
### 8.1 Train again on the full dataset

Refit the selected model family on all training rows and apply the validation-tuned threshold to the test predictions.

In [ ]:
X_full = train_features.drop(columns=[TARGET_COL])
y_full = train_features[TARGET_COL].astype(str)
preprocessor_full, _, _ = build_preprocessor(X_full)

final_models = {}
for name, original_model in models.items():
    if name not in trained_models:
        continue
    print("Refit", name)
    final_estimator = Pipeline([
        ("preprocess", preprocessor_full),
        ("model", original_model),
    ])
    try:
        final_estimator.fit(X_full, y_full)
    except Exception as exc:
        print("Skip failed final refit", name, "->", repr(exc))
        continue
    final_models[name] = final_estimator

if not final_models:
    raise RuntimeError("No final model was fitted.")

if best_model_name == "blend_mean":
    test_score_matrix = np.vstack([positive_probability(final_models[name], test_features) for name in final_models])
    test_scores = test_score_matrix.mean(axis=0)
    final_threshold = blend_threshold
else:
    best_row = report_df[report_df["model"] == best_model_name].iloc[0]
    test_scores = positive_probability(final_models[best_model_name], test_features)
    final_threshold = float(best_row["best_threshold"])

y_pred = np.where(test_scores >= final_threshold, POSITIVE_LABEL, NEGATIVE_LABEL)
print("Final threshold:", final_threshold)
print("Prediction distribution:")
display(pd.Series(y_pred).value_counts())

# 9. Prediction & Submission Generation
### 9.1 Build the final submission

Use `sample_submission.csv` as the template so row order, column names, and label format match the competition requirement.

In [ ]:
def infer_submission_label_column(sample_df: pd.DataFrame):
    if TARGET_COL in sample_df.columns:
        return TARGET_COL
    if "labels" in sample_df.columns:
        return "labels"
    if "label" in sample_df.columns:
        return "label"
    if "target" in sample_df.columns:
        return "target"
    return sample_df.columns[-1]


def format_predictions_for_submission(labels, sample_series: pd.Series):
    non_null = sample_series.dropna()
    if len(non_null) == 0:
        return labels
    sample_values = set(non_null.astype(str).str.strip().str.lower().unique())
    if sample_values and sample_values.issubset({"0", "1", "0.0", "1.0"}):
        return np.where(np.asarray(labels) == POSITIVE_LABEL, 1, 0)
    return labels

submission = sample_sub.copy()
submission_label_col = infer_submission_label_column(submission)
submission[submission_label_col] = format_predictions_for_submission(y_pred, submission[submission_label_col])

assert len(submission) == len(sample_sub), "row count mismatch"
assert submission[submission_label_col].notna().all(), "missing predictions"
if submission.columns[0] in sample_sub.columns:
    assert submission[submission.columns[0]].astype(str).tolist() == sample_sub[sample_sub.columns[0]].astype(str).tolist(), "id order changed"

submission.to_csv(SUBMISSION_PATH, index=False)
print("Submission saved to", SUBMISSION_PATH)
print("Submission shape:", submission.shape)
display(submission.head())
display(submission[submission_label_col].value_counts())